# Walmart Store Sales Forecasting — Prophet (კლასიკური მოდელი)

**ლოგირება:** Weights & Biases — პროექტი `ML-Final`, group `Prophet_Training`.

**რა არის Prophet** (Taylor & Letham, Facebook, 2017): დეკომპოზიციური additive მოდელი
`y(t) = g(t) + s(t) + h(t) + ε` — piecewise-linear **ტრენდი** + ფურიეს რიგებით აგებული **სეზონურობა** + **დღესასწაულების** ეფექტები. ფიტდება MAP ესტიმაციით (Stan). ძლიერი მხარეები: დღესასწაულების ექსპლიციტური მოდელირება, გამძლეობა გამოტოვებული მონაცემების მიმართ, ინტერპრეტირებადობა. სუსტი: უნივარიატულია — თითო სერიას ცალკე უყურებს, სერიებს შორის სწავლა (cross-learning) არ შეუძლია, ეგზოგენური ცხრილური ფიჩერების (Size, MarkDown...) ბუნებრივი მხარდაჭერა არ აქვს. სწორედ ამიტომ ველოდებით tree მოდელებზე სუსტ შედეგს — დავალების ინსტრუქციითაც კლასიკური მოდელები „თეორიულად გასარჩევადაა" და დიდ სატრენინგო დროს არ იმსახურებს.

**სტრატეგია:** 3 331 ცალკეული Prophet-ის ფიტი ~1.5 საათია და წვრილ სერიებზე ხმაურიანიც. ამის ნაცვლად:
1. Prophet ფიტდება **დეპარტამენტის ჯამურ სერიებზე** (81 ფიტი) — აგრეგატები გლუვია და Prophet-ის დეკომპოზიციას უხდება
2. პროგნოზი store-დონეზე ჩაიშლება **ისტორიული წილებით** (store-ის საშუალო წილი დეპარტამენტის ჯამში)

Run-ები: `Prophet_Preprocessing` → `Prophet_candidate_{i}` (baseline / +holidays / +multiplicative) → `Prophet_granularity_TypeDept` (უფრო წვრილი ჭრილი: Type×Dept ≈ 240 სერია) → `Prophet_Final_Pipeline` (pipeline პირდაპირ raw test-ზე, artifact-ად შენახული).

> Runtime: **CPU საკმარისია** — Prophet Stan-ზე ფიტდება და GPU-ს საერთოდ არ იყენებს. ჯამში ~15-25 წთ.

In [1]:
!pip install -q wandb prophet

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import wandb

wandb.login()  # API key: https://wandb.ai/authorize

WANDB_PROJECT = 'ML-Final'  # გუნდის საერთო პროექტი
GROUP = 'Prophet_Training'  # "ექსპერიმენტი" ამ არქიტექტურისთვის — ყველა run ამ ჯგუფში ერთიანდება
NB_VERSION = 'v1-dept-level'  # ჩაიწერება run-ების config-ში, რომ wandb-ზე ვერსიები გაიფილტროს

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: ERROR Invalid API key: API key may only contain the letters A-Z, digits and underscores.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


...

wandb: WARNING Invalid choice


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dshon23 (dshon23-free-university-of-tbilisi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
import numpy as np
import pandas as pd

BASE = '/content/drive/MyDrive/ML Final/Data/Raw/walmart-recruiting-store-sales-forecasting/'

train = pd.read_csv(BASE + 'train.csv')
test = pd.read_csv(BASE + 'test.csv')
features = pd.read_csv(BASE + 'features.csv')
stores = pd.read_csv(BASE + 'stores.csv')

for d in (train, test, features):
    d['Date'] = pd.to_datetime(d['Date'])

RAW_COLS = ['Store', 'Dept', 'Date', 'IsHoliday']  # ზუსტად ის სვეტები, რაც raw test.csv-შია
VAL_CUTOFF = '2012-08-17'  # იგივე time split, რაც გუნდმა EDA_Preprocessing-ში გამოიყენა


def wmae(y_true, y_pred, is_holiday):
    """Competition metric: MAE, სადაც სადღესასწაულო კვირებს წონა 5 აქვთ."""
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w)


print(train.shape, test.shape, features.shape, stores.shape)

(421570, 5) (115064, 4) (8190, 12) (45, 3)


## Run 1 — Preprocessing

- უარყოფითი sales → 0 (გუნდის EDA-ს გადაწყვეტილება)
- დეპარტამენტის ჯამური სერიები: `y_dept(t) = Σ_stores Weekly_Sales`
- store-ის წილი დეპარტამენტში დისაგრეგაციისთვის; ვალიდაციისას წილები **მხოლოდ** pre-cutoff მონაცემით ითვლება (leakage არ არის)

In [5]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='Prophet_Preprocessing',
                 job_type='preprocessing',
                 config={'strategy': 'aggregate-level Prophet + store-share disaggregation',
                         'negative_sales_strategy': 'clip_to_zero',
                         'granularities': 'Dept (81 series) vs Type x Dept (~240 series)'})

train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

# Type დაგვჭირდება granularity ექსპერიმენტისთვის (A/B/C მაღაზიები სხვანაირად იქცევიან)
train = train.merge(stores[['Store', 'Type']], on='Store', how='left')
test = test.merge(stores[['Store', 'Type']], on='Store', how='left')


def make_series(df, group_cols):
    """ჯამური კვირეული სერიები მოცემულ ჭრილში, Prophet-ის ფორმატში (ds, y)."""
    return (df.groupby(group_cols + ['Date'])['Weekly_Sales'].sum()
              .rename('y').reset_index().rename(columns={'Date': 'ds'}))


def make_shares(df, group_cols):
    """store-dept წყვილის ისტორიული წილი ჯგუფის ჯამურ გაყიდვებში."""
    key_cols = list(dict.fromkeys(['Store', 'Dept'] + group_cols))  # უნიკალური სვეტები
    s = (df.groupby(key_cols)['Weekly_Sales'].mean()
           .rename('sd_mean').reset_index())
    s['share'] = s['sd_mean'] / s.groupby(group_cols)['sd_mean'].transform('sum')
    return s


run.log({'n_dept_series': int(train['Dept'].nunique()),
         'n_typedept_series': int(train.groupby(['Type', 'Dept']).ngroups),
         'n_store_dept_pairs': int(train.groupby(['Store', 'Dept']).ngroups)})
run.finish()

n_dept_series,▁
n_store_dept_pairs,▁
n_typedept_series,▁
n_dept_series,81
n_store_dept_pairs,3331
n_typedept_series,227


In [6]:
from prophet import Prophet
import logging

for name in ('prophet', 'cmdstanpy'):
    logging.getLogger(name).setLevel(logging.ERROR)

# კონკურსის 4 დღესასწაული, train+test პერიოდის ზუსტი კვირის თარიღებით
HOLIDAYS = pd.DataFrame(
    [{'holiday': 'SuperBowl', 'ds': d} for d in
     ['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08']] +
    [{'holiday': 'LaborDay', 'ds': d} for d in
     ['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06']] +
    [{'holiday': 'Thanksgiving', 'ds': d} for d in
     ['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29']] +
    [{'holiday': 'Christmas', 'ds': d} for d in
     ['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27']])
HOLIDAYS['ds'] = pd.to_datetime(HOLIDAYS['ds'])


def fit_forecast_one(g, horizon_dates, cfg):
    """ერთ ჯამურ სერიაზე Prophet-ის ფიტი და პროგნოზი horizon_dates-ზე."""
    if len(g) < 30 or g['y'].sum() <= 0:  # ძალიან მოკლე/მკვდარი სერია -> საშუალო
        return np.full(len(horizon_dates), max(float(g['y'].mean()), 0.0))
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                holidays=cfg['holidays'], seasonality_mode=cfg['seasonality_mode'])
    m.fit(g[['ds', 'y']])
    yhat = m.predict(pd.DataFrame({'ds': horizon_dates}))['yhat'].values
    return np.clip(yhat, 0, None)


def forecast_all(series, horizon_dates, cfg, group_cols):
    """ყველა ჯამური სერიის პროგნოზი: dataframe (group_cols..., Date, group_pred)."""
    rows = []
    for i, (key, g) in enumerate(series.groupby(group_cols)):
        key = key if isinstance(key, tuple) else (key,)
        block = pd.DataFrame({'Date': horizon_dates,
                              'group_pred': fit_forecast_one(g, horizon_dates, cfg)})
        for col, v in zip(group_cols, key):
            block[col] = v
        rows.append(block)
        if (i + 1) % 25 == 0:
            print(f'  {i + 1} series done')
    return pd.concat(rows, ignore_index=True)


val_actual = train[train['Date'] >= VAL_CUTOFF][['Store', 'Dept', 'Date', 'IsHoliday', 'Weekly_Sales']]
val_dates = pd.DatetimeIndex(np.sort(val_actual['Date'].unique()))


def evaluate(cfg, group_cols):
    """fit pre-cutoff მონაცემზე მოცემულ ჭრილში -> holdout WMAE (იგივე პროტოკოლი, რაც ყველგან)."""
    tr = train[train['Date'] < VAL_CUTOFF]
    gpred = forecast_all(make_series(tr, group_cols), val_dates, cfg, group_cols)
    long = make_shares(tr, group_cols).merge(gpred, on=group_cols, how='inner')
    long['pred'] = (long['group_pred'] * long['share']).clip(lower=0)
    m = val_actual.merge(long[['Store', 'Dept', 'Date', 'pred']],
                         on=['Store', 'Dept', 'Date'], how='left')
    m['pred'] = m['pred'].fillna(0)
    return float(wmae(m['Weekly_Sales'], m['pred'], m['IsHoliday']))


print('validation weeks:', len(val_dates))

validation weeks: 11


## Run 2 — კანდიდატები

სამი კონფიგურაცია (დავალების რეკომენდაციით — მსუბუქად, დიდი ძებნის გარეშე):
1. **baseline** — მხოლოდ ტრენდი + წლიური სეზონურობა
2. **+holidays** — კონკურსის 4 დღესასწაული ექსპლიციტურად (Prophet-ის მთავარი კოზირი ამ ამოცანაზე)
3. **+multiplicative** — სეზონურობა დონის პროპორციულია (გაყიდვების პიკები დონესთან ერთად იზრდება)

ვალიდაცია: ზუსტად იგივე holdout (`>= 2012-08-17`) და WMAE, რაც ყველა სხვა არქიტექტურაში.

In [7]:
CANDIDATES = [
    dict(name='baseline', holidays=None, seasonality_mode='additive'),
    dict(name='holidays', holidays=HOLIDAYS, seasonality_mode='additive'),
    dict(name='holidays_multiplicative', holidays=HOLIDAYS, seasonality_mode='multiplicative'),
]

best_val, BEST_CFG = np.inf, None
for i, cfg in enumerate(CANDIDATES):
    run = wandb.init(project=WANDB_PROJECT, group=GROUP, name=f'Prophet_candidate_{i}',
                     job_type='hyperparam_search',
                     config={'name': cfg['name'], 'seasonality_mode': cfg['seasonality_mode'],
                             'holidays': cfg['holidays'] is not None,
                             'granularity': 'Dept', 'features_version': NB_VERSION})
    print(f"candidate {i}: {cfg['name']}")
    score = evaluate(cfg, ['Dept'])
    run.summary['val_wmae'] = score
    run.finish()
    print(f'  -> val WMAE {score:,.2f}')
    if score < best_val:
        best_val, BEST_CFG = score, cfg

print('Best config:', BEST_CFG['name'], f'-> {best_val:,.2f}')

candidate 0: baseline
  25 series done
  50 series done
  75 series done


val_wmae,1904.16467


  -> val WMAE 1,904.16


candidate 1: holidays
  25 series done
  50 series done
  75 series done


val_wmae,1840.31324


  -> val WMAE 1,840.31


candidate 2: holidays_multiplicative
  25 series done
  50 series done
  75 series done


val_wmae,1839.26091


  -> val WMAE 1,839.26
Best config: holidays_multiplicative -> 1,839.26


## Run 3 — Granularity ექსპერიმენტი

დამატებითი კვლევითი კითხვა: **რომელ ჭრილში ჯობია Prophet-ის ფიტი?**
- `Dept` (81 სერია) — გლუვი აგრეგატები, მაგრამ A/B/C ტიპის მაღაზიების განსხვავებული დინამიკა იკარგება
- `Type × Dept` (~240 სერია) — მეტი დეტალი, მაგრამ თითო სერია უფრო ხმაურიანია

საუკეთესო კონფიგურაციით ორივეს ვადარებთ იმავე holdout-ზე.

In [8]:
run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='Prophet_granularity_TypeDept',
                 job_type='hyperparam_search',
                 config={'name': BEST_CFG['name'], 'granularity': 'Type_Dept',
                         'features_version': NB_VERSION})
print('granularity: Type x Dept')
score_td = evaluate(BEST_CFG, ['Type', 'Dept'])
run.summary['val_wmae'] = score_td
run.finish()
print(f'Type x Dept -> val WMAE {score_td:,.2f}  (Dept-level: {best_val:,.2f})')

if score_td < best_val:
    BEST_GRAN, val_score = ['Type', 'Dept'], score_td
else:
    BEST_GRAN, val_score = ['Dept'], best_val
print('Selected granularity:', BEST_GRAN, f'-> {val_score:,.2f}')

granularity: Type x Dept
  25 series done
  50 series done
  75 series done
  100 series done
  125 series done
  150 series done
  175 series done
  200 series done
  225 series done


val_wmae,1831.40462


Type x Dept -> val WMAE 1,831.40  (Dept-level: 1,839.26)
Selected granularity: ['Type', 'Dept'] -> 1,831.40


## Run 4 — Final Pipeline

საუკეთესო კონფიგურაცია ფიტდება **მთელ** ისტორიაზე, პროგნოზირებს test-ის 39 კვირას დეპარტამენტების დონეზე და წილებით ჩაიშლება store-dept-ზე. `ProphetForecastPipeline` (იგივე შაბლონი, რაც DLinear-ში) **raw** test dataframe-ს იღებს — preprocessing არ სჭირდება; ინახება W&B Artifact-ად.

In [9]:
import cloudpickle

test_dates = pd.DatetimeIndex(np.sort(test['Date'].unique()), name='Date')
print('test horizon:', len(test_dates), 'weeks')


class ProphetForecastPipeline:
    """End-to-end pipeline: raw test rows (Store, Dept, Date, IsHoliday) -> predictions.

    ინახავს dept-დონის Prophet პროგნოზების store-წილებით ჩაშლილ ცხრილს + fallback
    საშუალოებს, ამიტომ predict() პირდაპირ დაუმუშავებელ test set-ზე მუშაობს.
    """

    def __init__(self, forecast_long, sd_mean, dept_mean, global_mean):
        self.forecast_long = forecast_long
        self.sd_mean = sd_mean
        self.dept_mean = dept_mean
        self.global_mean = global_mean

    def predict(self, model_input):
        df = model_input.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        out = (df.merge(self.forecast_long, on=['Store', 'Dept', 'Date'], how='left')
                 .merge(self.sd_mean, on=['Store', 'Dept'], how='left')
                 .merge(self.dept_mean, on='Dept', how='left'))
        pred = (out['pred'].fillna(out['SD_Mean'])
                           .fillna(out['Dept_Mean'])
                           .fillna(self.global_mean))
        return pred.clip(lower=0).values


run = wandb.init(project=WANDB_PROJECT, group=GROUP, name='Prophet_Final_Pipeline',
                 job_type='final_pipeline',
                 config={'name': BEST_CFG['name'], 'seasonality_mode': BEST_CFG['seasonality_mode'],
                         'holidays': BEST_CFG['holidays'] is not None,
                         'granularity': '_'.join(BEST_GRAN),
                         'features_version': NB_VERSION})
run.summary['val_wmae'] = val_score  # შეფასება კანდიდატების holdout-იდან

print('fitting on full history...')
gpred_test = forecast_all(make_series(train, BEST_GRAN), test_dates, BEST_CFG, BEST_GRAN)
shares_full = make_shares(train, BEST_GRAN)

forecast_long = shares_full.merge(gpred_test, on=BEST_GRAN, how='inner')
forecast_long['pred'] = (forecast_long['group_pred'] * forecast_long['share']).clip(lower=0)
forecast_long = forecast_long[['Store', 'Dept', 'Date', 'pred']]

sd_mean = (train.groupby(['Store', 'Dept'])['Weekly_Sales']
                .mean().rename('SD_Mean').reset_index())
dept_mean = (train.groupby('Dept')['Weekly_Sales']
                  .mean().rename('Dept_Mean').reset_index())

pipeline = ProphetForecastPipeline(forecast_long, sd_mean, dept_mean,
                                   float(train['Weekly_Sales'].mean()))

with open('prophet_pipeline.pkl', 'wb') as f:
    cloudpickle.dump(pipeline, f)

artifact = wandb.Artifact('prophet_pipeline', type='model',
                          description='Aggregate-level Prophet + store-share disaggregation '
                                      '(runs directly on the raw test set)',
                          metadata={'val_wmae': float(val_score), 'config': BEST_CFG['name'],
                                    'granularity': '_'.join(BEST_GRAN)})
artifact.add_file('prophet_pipeline.pkl')
run.log_artifact(artifact)

# pipeline პირდაპირ RAW test set-ზე
test_pred = pipeline.predict(test[RAW_COLS])
submission = pd.DataFrame({
    'Id': (test['Store'].astype(str) + '_' + test['Dept'].astype(str)
           + '_' + test['Date'].dt.strftime('%Y-%m-%d')),
    'Weekly_Sales': test_pred,
})
submission.to_csv('submission_prophet.csv', index=False)
sub_art = wandb.Artifact('prophet_submission', type='submission')
sub_art.add_file('submission_prophet.csv')
run.log_artifact(sub_art)

run.finish()
submission.head()

test horizon: 39 weeks


fitting on full history...
  25 series done
  50 series done
  75 series done
  100 series done
  125 series done
  150 series done
  175 series done
  200 series done
  225 series done


val_wmae,1831.40462


,Id,Weekly_Sales
0,1_1_2012-11-02,37354.977915
1,1_1_2012-11-09,29244.552602
2,1_1_2012-11-16,18340.299306
3,1_1_2012-11-23,14699.986654
4,1_1_2012-11-30,21440.008072


In [10]:
# ვინახავთ საუკეთესო pipeline-ის artifact-ის სახელს Drive-ზე, რომ model_inference.ipynb-მ
# ყველა არქიტექტურა შეადაროს და საუკეთესო W&B Model Registry-ში დაალინკოს.
import json, os

reg_path = '/content/drive/MyDrive/ML Final/best_runs.json'
info = json.load(open(reg_path)) if os.path.exists(reg_path) else {}
info['Prophet'] = {
    'artifact': f'{WANDB_PROJECT}/prophet_pipeline:latest',
    'val_wmae': float(val_score),
}
with open(reg_path, 'w') as f:
    json.dump(info, f, indent=2)
info

{'XGBoost': {'artifact': 'ML-Final/xgboost_pipeline:latest',
  'val_wmae': 1451.361784955096},
 'LightGBM': {'artifact': 'ML-Final/lightgbm_pipeline:latest',
  'val_wmae': 1508.4106169389534},
 'DLinear': {'artifact': 'ML-Final/dlinear_pipeline:latest',
  'val_wmae': 1602.6865936618267},
 'NBEATS': {'artifact': 'ML-Final/nbeats_pipeline:latest',
  'val_wmae': 1476.8615020275404},
 'PatchTST': {'artifact': 'ML-Final/patchtst_pipeline:latest',
  'val_wmae': 1554.1872775576571},
 'Prophet': {'artifact': 'ML-Final/prophet_pipeline:latest',
  'val_wmae': 1831.404616769137}}

In [11]:
# sanity check: pipeline W&B artifact-იდან ჩამოვტვირთოთ და raw test-ზე გავუშვათ
import cloudpickle, os

api = wandb.Api()
art = api.artifact(f'{WANDB_PROJECT}/prophet_pipeline:latest')
art_dir = art.download()
with open(os.path.join(art_dir, 'prophet_pipeline.pkl'), 'rb') as f:
    loaded = cloudpickle.load(f)
print(loaded.predict(test[RAW_COLS].head())[:5])

wandb:   1 of 1 files downloaded.  


[37354.97791455 29244.55260222 18340.29930615 14699.98665445
 21440.00807192]


## შემდეგი ნაბიჯი

- შედეგი ჩაიწერა `best_runs.json`-ში — `model_inference.ipynb` მას სხვა არქიტექტურებს შეადარებს.
- README-სთვის ჩაინიშნეთ: რომელმა კონფიგურაციამ მოიგო (holidays-ის ეფექტი განსაკუთრებით საინტერესოა — ეს Prophet-ის მთავარი უპირატესობაა ამ ამოცანაზე) და სად დგას კლასიკური მოდელი trees/DL-თან შედარებით.
- მოსალოდნელი შედეგი: val WMAE ~1 900–2 400 — tree მოდელებზე სუსტი, რადგან (ა) უნივარიატულია, cross-learning არ აქვს; (ბ) დისაგრეგაცია წილებით store-სპეციფიკურ დინამიკას კარგავს; (გ) ეგზოგენური ფიჩერები (MarkDown, Size...) არ ესმის. ეს არ არის მარცხი — ეს არის კლასიკური კატეგორიის დოკუმენტირებული პასუხი.